# POST-TRAINING QUANTIZATION (PTQ)

    This notebook demonstrates POST-TRAINING QUANTIZATION on a real neural network.
    
    KEY CONCEPT: Post-Training Quantization (PTQ)
    - Train your model normally in FP32 (32-bit floating point)
    - AFTER training, convert the model to INT8 (8-bit integers)
    - Use a calibration dataset to determine optimal quantization parameters
    - No retraining required!
    
    WORKFLOW:
    1. Train model (FP32) → 2. Calibrate (observe data) → 3. Quantize (INT8)
    
    BENEFITS:
     Smaller model size (32-bit → 8-bit)
     Faster inference (integer math is faster than float)
     No retraining needed (unlike Quantization-Aware Training)
     Usually <1% accuracy loss

## LOAD MNIST DATASET

In [1]:
import torch
import torchvision.datasets as datasets 
import torchvision.transforms as transforms
import torch.nn as nn
import matplotlib.pyplot as plt
from tqdm import tqdm
from pathlib import Path
import os

# Make torch deterministic for reproducible results
_ = torch.manual_seed(0)

# Normalize MNIST images using dataset mean and std
# Mean: 0.1307, Std: 0.3081 (standard MNIST normalization)
transform = transforms.Compose([
    transforms.ToTensor(),  # Convert PIL image to tensor [0, 1]
    transforms.Normalize((0.1307,), (0.3081,))  # Normalize to ~N(0,1)
])

# Load MNIST training set
mnist_trainset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
train_loader = torch.utils.data.DataLoader(mnist_trainset, batch_size=10, shuffle=True)

# Load MNIST test set
mnist_testset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)
test_loader = torch.utils.data.DataLoader(mnist_testset, batch_size=10, shuffle=True)

# Use cuda/CPU for this demo
device = "cpu"

## DEFINE THE NEURAL NETWORK MODEL

In [2]:
class VerySimpleNet(nn.Module):
    """
    A simple 3-layer fully-connected neural network.
    
    Architecture:
    Input (784) -> Linear1 (100) -> ReLU -> Linear2 (100) -> ReLU -> Linear3 (10) -> Output
    
    Visual representation:
    
    [28×28 image]
         ↓ (flatten to 784)
    [784 neurons] ──────────> [100 neurons] (hidden layer 1)
         ↓                         ↓
    Linear1 + ReLU            78,400 weights
                                   ↓
                              [100 neurons] (hidden layer 2)
                                   ↓
                              Linear2 + ReLU
                                   ↓
                              [10 neurons] (output)
                                   ↓
                              Linear3
                                   ↓
                         [logits for classes 0-9]
    
    Total parameters: 78,400 + 10,000 + 1,000 = 89,400 weights
    Size in FP32: 89,400 × 4 bytes = 357.6 KB
    """
    def __init__(self, hidden_size_1=100, hidden_size_2=100):
        super(VerySimpleNet, self).__init__()
        # Layer 1: 784 inputs → 100 outputs
        self.linear1 = nn.Linear(28*28, hidden_size_1) 
        # Layer 2: 100 inputs → 100 outputs
        self.linear2 = nn.Linear(hidden_size_1, hidden_size_2) 
        # Layer 3: 100 inputs → 10 outputs (one per digit class)
        self.linear3 = nn.Linear(hidden_size_2, 10)
        # ReLU activation: max(0, x) - introduces non-linearity
        self.relu = nn.ReLU()

    def forward(self, img):
        # Flatten 28×28 image to 784 vector
        x = img.view(-1, 28*28)
        # Hidden layer 1 with activation
        x = self.relu(self.linear1(x))
        # Hidden layer 2 with activation
        x = self.relu(self.linear2(x))
        # Output layer (no activation, raw logits)
        x = self.linear3(x)
        return x

# Create the model instance
net = VerySimpleNet().to(device)

## TRAINING FUNCTIONS

In [3]:
def train(train_loader, net, epochs=5, total_iterations_limit=None):
    """
    Standard training loop using cross-entropy loss and Adam optimizer.
    """
    cross_el = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(net.parameters(), lr=0.001)

    total_iterations = 0

    for epoch in range(epochs):
        net.train()  # Set model to training mode

        loss_sum = 0
        num_iterations = 0

        data_iterator = tqdm(train_loader, desc=f'Epoch {epoch+1}')
        if total_iterations_limit is not None:
            data_iterator.total = total_iterations_limit
            
        for data in data_iterator:
            num_iterations += 1
            total_iterations += 1
            x, y = data
            x = x.to(device)
            y = y.to(device)
            
            # Standard training step
            # Clear gradients from previous iteration to prevent accumulation
            optimizer.zero_grad()
            
            # Forward pass: reshape input images from (batch, 28, 28) to (batch, 784)
            # and pass through the network to get predictions
            output = net(x.view(-1, 28*28))
            
            # Calculate loss: compare network predictions with true labels
            # using cross-entropy loss (cross_el is the loss function)
            loss = cross_el(output, y)
            
            # Accumulate loss for tracking average loss over time
            loss_sum += loss.item()  # .item() extracts scalar value from tensor
            
            # Calculate running average loss across all iterations so far
            avg_loss = loss_sum / num_iterations
            
            # Update progress bar display with current average loss
            data_iterator.set_postfix(loss=avg_loss)
            
            # Backward pass: compute gradients of loss with respect to all parameters
            loss.backward()
            
            # Update network parameters using computed gradients
            # applies the optimization algorithm
            optimizer.step()

            if total_iterations_limit is not None and total_iterations >= total_iterations_limit:
                return

            
def print_size_of_model(model):
    """
    Print the size of the model in KB by saving and checking file size.
    """
    torch.save(model.state_dict(), "temp_delme.p")
    print('Size (KB):', os.path.getsize("temp_delme.p")/1e3)
    os.remove('temp_delme.p')

In [4]:
# Train the model or load from disk if already trained
MODEL_FILENAME = 'simplenet_ptq.pt'

if Path(MODEL_FILENAME).exists():
    net.load_state_dict(torch.load(MODEL_FILENAME))
    print('Loaded model from disk')
else:
    train(train_loader, net, epochs=1)
    torch.save(net.state_dict(), MODEL_FILENAME)

Epoch 1: 100%|████████████████████████████████████████████████████████| 6000/6000 [00:26<00:00, 223.63it/s, loss=0.223]


## TESTING FUNCTION

In [5]:
def test(model: nn.Module, total_iterations: int = None):
    """
    Evaluate model accuracy on test set.
    """
    correct = 0
    total = 0
    iterations = 0

    model.eval()  # Set model to evaluation mode

    with torch.no_grad():  # No gradient computation needed
        for data in tqdm(test_loader, desc='Testing'):
            x, y = data
            x = x.to(device)
            y = y.to(device)
            output = model(x.view(-1, 784))
            
            # Count correct predictions
            for idx, i in enumerate(output):
                if torch.argmax(i) == y[idx]:
                    correct += 1
                total += 1
                
            iterations += 1
            if total_iterations is not None and iterations >= total_iterations:
                break
                
    print(f'Accuracy: {round(correct/total, 3)}')

## BASELINE MODEL STATISTICS (BEFORE QUANTIZATION)

In [6]:
print('Weights before quantization')
print(net.linear1.weight)
print(net.linear1.weight.dtype)

print('Size of the model before quantization')
print_size_of_model(net)

print(f'Accuracy of the model before quantization: ')
test(net)

Weights before quantization
Parameter containing:
tensor([[ 0.0005,  0.0199, -0.0286,  ...,  0.0227,  0.0045,  0.0028],
        [-0.0224, -0.0176, -0.0131,  ..., -0.0229, -0.0086, -0.0326],
        [ 0.0304,  0.0654,  0.0173,  ...,  0.0302,  0.0517,  0.0586],
        ...,
        [ 0.0586,  0.0623,  0.0277,  ...,  0.0223,  0.0416,  0.0046],
        [-0.0069,  0.0012,  0.0332,  ...,  0.0135,  0.0136,  0.0131],
        [ 0.0016, -0.0037, -0.0179,  ...,  0.0186, -0.0307, -0.0106]],
       requires_grad=True)
torch.float32
Size of the model before quantization
Size (KB): 360.998
Accuracy of the model before quantization: 


Testing: 100%|████████████████████████████████████████████████████████████████████| 1000/1000 [00:01<00:00, 511.12it/s]

Accuracy: 0.959


## CREATE QUANTIZATION-READY MODEL

In [7]:
class QuantizedVerySimpleNet(nn.Module):
    """
    Same network as before, but with quantization stubs added.
    
    KEY ADDITIONS:
    1. QuantStub: Marks where to quantize inputs (FP32 → INT8)
    2. DeQuantStub: Marks where to dequantize outputs (INT8 → FP32)
    
    These stubs tell PyTorch:
    - "Convert inputs to INT8 here"
    - "Convert outputs back to FP32 here"
    - "Everything in between runs in INT8"
    
    Visual flow:
    
    FP32 Input -> [QuantStub] -> INT8 -> Network (INT8 ops) -> INT8 -> [DeQuantStub] -> FP32 Output
                     ↓                                                    ↓
              Quantize to INT8                                   Dequantize to FP32
    """
    def __init__(self, hidden_size_1=100, hidden_size_2=100):
        super(QuantizedVerySimpleNet, self).__init__()
        # NEW: Quantization stub (input quantization)
        self.quant = torch.quantization.QuantStub()
        
        self.linear1 = nn.Linear(28*28, hidden_size_1) 
        self.linear2 = nn.Linear(hidden_size_1, hidden_size_2) 
        self.linear3 = nn.Linear(hidden_size_2, 10)
        self.relu = nn.ReLU()
        
        # NEW: Dequantization stub (output dequantization)
        self.dequant = torch.quantization.DeQuantStub()

    def forward(self, img):
        x = img.view(-1, 28*28)
        # NEW: Quantize input to INT8
        x = self.quant(x)
        # Network operations (will run in INT8 after quantization)
        x = self.relu(self.linear1(x))
        x = self.relu(self.linear2(x))
        x = self.linear3(x)
        # NEW: Dequantize output back to FP32
        x = self.dequant(x)
        return x

## PREPARE MODEL FOR QUANTIZATION (INSERT OBSERVERS)

In [8]:
# Create quantized model and copy trained weights
net_quantized = QuantizedVerySimpleNet().to(device)
net_quantized.load_state_dict(net.state_dict())
net_quantized.eval()

# Set quantization configuration (use PyTorch defaults)
net_quantized.qconfig = torch.ao.quantization.default_qconfig

# CRITICAL STEP: Insert observers
net_quantized = torch.ao.quantization.prepare(net_quantized)
print(net_quantized)

QuantizedVerySimpleNet(
  (quant): QuantStub(
    (activation_post_process): MinMaxObserver(min_val=inf, max_val=-inf)
  )
  (linear1): Linear(
    in_features=784, out_features=100, bias=True
    (activation_post_process): MinMaxObserver(min_val=inf, max_val=-inf)
  )
  (linear2): Linear(
    in_features=100, out_features=100, bias=True
    (activation_post_process): MinMaxObserver(min_val=inf, max_val=-inf)
  )
  (linear3): Linear(
    in_features=100, out_features=10, bias=True
    (activation_post_process): MinMaxObserver(min_val=inf, max_val=-inf)
  )
  (relu): ReLU()
  (dequant): DeQuantStub()
)


    PyTorch inserted MinMaxObserver objects
    
    MinMaxObserver:
    ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    - Tracks min and max values during inference
    - Records: min_val=inf, max_val=-inf (initialized, not yet calibrated)
    - Will observe actual data during calibration phase
    
    Purpose: Determine the optimal scale and zero-point for each layer!
    
    Visual representation of what's happening:
        
        Input -> [Observer] -> Linear1 -> [Observer] -> ReLU -> Linear2 -> [Observer] -> ...
                   ↓                       ↓                             ↓
             Track min/max           Track min/max               Track min/max
             
    Each observer will record the range of activations to calculate
    the best quantization parameters (scale, zero_point).

## CALIBRATION - RUN DATA THROUGH OBSERVERS

In [9]:
print("Running calibration on test set...")
test(net_quantized)

"""
During this test run, the observers are COLLECTING STATISTICS:

For each layer, they track:
- Minimum activation value seen
- Maximum activation value seen

Example for linear1 layer:
    Batch 1: sees values from -45.2 to 30.1
    Batch 2: sees values from -50.3 to 34.8
    Batch 3: sees values from -53.5 to 32.4
    ...
    Final: min_val = -53.58, max_val = 34.90

These statistics will be used to compute:
    scale = (max - min) / 255
    zero_point = -round(min / scale)

The model is still running in FP32, just observing!
"""

# Check the collected statistics
print(f'Check statistics of the various layers')
print(net_quantized)

Running calibration on test set...


Testing: 100%|████████████████████████████████████████████████████████████████████| 1000/1000 [00:02<00:00, 464.35it/s]

Accuracy: 0.959
Check statistics of the various layers
QuantizedVerySimpleNet(
  (quant): QuantStub(
    (activation_post_process): MinMaxObserver(min_val=-0.4242129623889923, max_val=2.821486711502075)
  )
  (linear1): Linear(
    in_features=784, out_features=100, bias=True
    (activation_post_process): MinMaxObserver(min_val=-51.221126556396484, max_val=38.080013275146484)
  )
  (linear2): Linear(
    in_features=100, out_features=100, bias=True
    (activation_post_process): MinMaxObserver(min_val=-23.95867919921875, max_val=25.460731506347656)
  )
  (linear3): Linear(
    in_features=100, out_features=10, bias=True
    (activation_post_process): MinMaxObserver(min_val=-24.751096725463867, max_val=19.82545280456543)
  )
  (relu): ReLU()
  (dequant): DeQuantStub()
)


Example output breakdown:

    MinMaxObserver(min_val=-0.424, max_val=2.821)  ← Input layer
        Range: 3.245 units
        These are the normalized input pixel values after preprocessing
    
    MinMaxObserver(min_val=-53.584, max_val=34.898)  ← After linear1
        Range: 88.482 units  
        These are the activation values after first layer
    
    MinMaxObserver(min_val=-24.331, max_val=26.625)  ← After linear2
        Range: 50.956 units
        Activations after second layer
    
    Notice how each layer has different ranges - that's why we need
    per-layer quantization parameters!

## CONVERT MODEL TO QUANTIZED VERSION

In [10]:
net_quantized = torch.ao.quantization.convert(net_quantized)

    torch.ao.quantization.convert() does:
    1. Calculates scale and zero_point from observed min/max
    2. Converts all FP32 weights to INT8
    3. Replaces FP32 layers with quantized equivalents
    4. Removes observers (no longer needed)
    
    Transformation:
        nn.Linear (FP32) → QuantizedLinear (INT8)
        
    Before convert():
        Linear layer with FP32 weights: [-0.0068, 0.0126, -0.0359, ...]
        
    After convert():
        QuantizedLinear with INT8 weights: [-2, 3, -8, ...]
        + scale factor: 0.0043
        + zero_point: 0
        
    To get FP32 back: weight_fp32 ≈ (weight_int8 - zero_point) × scale

In [11]:
print(f'Check statistics of the various layers')
print(net_quantized)

Check statistics of the various layers
QuantizedVerySimpleNet(
  (quant): Quantize(scale=tensor([0.0256]), zero_point=tensor([17]), dtype=torch.quint8)
  (linear1): QuantizedLinear(in_features=784, out_features=100, scale=0.7031585574150085, zero_point=73, qscheme=torch.per_tensor_affine)
  (linear2): QuantizedLinear(in_features=100, out_features=100, scale=0.3891292214393616, zero_point=62, qscheme=torch.per_tensor_affine)
  (linear3): QuantizedLinear(in_features=100, out_features=10, scale=0.3509964644908905, zero_point=71, qscheme=torch.per_tensor_affine)
  (relu): ReLU()
  (dequant): DeQuantize()
)


    Quantize(scale=0.0256, zero_point=17)  ← Input quantization
        Converts FP32 inputs to INT8 using these parameters
    
    QuantizedLinear(..., scale=0.802, zero_point=79)  ← Layer 1
        All operations in INT8!
        - INT8 weights
        - INT8 activations  
        - INT8 matrix multiplication
        - Much faster
    
    QuantizedLinear(..., scale=0.588, zero_point=66)  ← Layer 2
    
    QuantizedLinear(..., scale=0.626, zero_point=88)  ← Layer 3
    
    DeQuantize()  ← Output dequantization
        Converts INT8 outputs back to FP32
    
    Each layer has its own scale/zero_point optimized for its range

## EXAMINE QUANTIZED WEIGHTS

In [12]:
print('Weights after quantization')
print(torch.int_repr(net_quantized.linear1.weight()))

Weights after quantization
tensor([[ 0,  4, -6,  ...,  5,  1,  1],
        [-5, -4, -3,  ..., -5, -2, -7],
        [ 7, 15,  4,  ...,  7, 12, 13],
        ...,
        [13, 14,  6,  ...,  5,  9,  1],
        [-2,  0,  7,  ...,  3,  3,  3],
        [ 0, -1, -4,  ...,  4, -7, -2]], dtype=torch.int8)


In [13]:
print('Original weights: ')
print(net.linear1.weight)
print('')
print(f'Dequantized weights: ')
print(torch.dequantize(net_quantized.linear1.weight()))
print('')

Original weights: 
Parameter containing:
tensor([[ 0.0005,  0.0199, -0.0286,  ...,  0.0227,  0.0045,  0.0028],
        [-0.0224, -0.0176, -0.0131,  ..., -0.0229, -0.0086, -0.0326],
        [ 0.0304,  0.0654,  0.0173,  ...,  0.0302,  0.0517,  0.0586],
        ...,
        [ 0.0586,  0.0623,  0.0277,  ...,  0.0223,  0.0416,  0.0046],
        [-0.0069,  0.0012,  0.0332,  ...,  0.0135,  0.0136,  0.0131],
        [ 0.0016, -0.0037, -0.0179,  ...,  0.0186, -0.0307, -0.0106]],
       requires_grad=True)

Dequantized weights: 
tensor([[ 0.0000,  0.0180, -0.0269,  ...,  0.0224,  0.0045,  0.0045],
        [-0.0224, -0.0180, -0.0135,  ..., -0.0224, -0.0090, -0.0314],
        [ 0.0314,  0.0673,  0.0180,  ...,  0.0314,  0.0539,  0.0584],
        ...,
        [ 0.0584,  0.0628,  0.0269,  ...,  0.0224,  0.0404,  0.0045],
        [-0.0090,  0.0000,  0.0314,  ...,  0.0135,  0.0135,  0.0135],
        [ 0.0000, -0.0045, -0.0180,  ...,  0.0180, -0.0314, -0.0090]])



## FINAL RESULTS - SIZE AND ACCURACY

In [14]:
print('Size of the model after quantization')
print_size_of_model(net_quantized)

Size of the model after quantization
Size (KB): 95.394


    Before quantization: ~360 KB
    After quantization:  ~95 KB
    
    Reduction: 360 → 95 KB = 73.6% smaller
    
    Why not exactly 75% (4x) reduction?
    - Scale and zero_point parameters take some space (FP32)
    - Model structure metadata
    - But still massive savings!
    
    Real-world impact:
    - Fits in smaller devices (mobile, embedded)
    - Faster to load over network
    - Less memory usage during inference
    - Faster computation (INT8 operations)

In [15]:
print('Testing the model after quantization')
test(net_quantized)

Testing the model after quantization


Testing: 100%|████████████████████████████████████████████████████████████████████| 1000/1000 [00:02<00:00, 431.40it/s]

Accuracy: 0.96


    Accuracy maintained
    
    This is the GOAL of good quantization:
     Massive size reduction (~75%)
     Minimal accuracy loss
     Faster inference (INT8 operations)
    
    In practice:
    - Simple networks: Usually <1% accuracy drop
    - Complex networks: 1-3% drop typical
    - Can use Quantization-Aware Training (QAT) if PTQ isn't enough

    POST-TRAINING QUANTIZATION (PTQ) - Complete Pipeline:
    ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    
    Step 1: Train Model (FP32)
        [Train normally] → [360 KB model] → [96% accuracy]
                                    ↓
    Step 2: Add Quantization Stubs
        Add QuantStub() and DeQuantStub() to mark quantization boundaries
                                    ↓
    Step 3: Insert Observers
        torch.quantization.prepare() adds MinMaxObservers
                                    ↓
    Step 4: Calibrate
        Run representative data through model to collect statistics
        Observers track min/max values for each layer
                                    ↓
    Step 5: Convert to INT8
        torch.quantization.convert() converts FP32 → INT8
        Calculates optimal scale/zero_point per layer
                                    ↓
    Step 6: Deploy!
        [95 KB model] → [96% accuracy] → [4x faster inference]
        
        
    WHEN TO USE PTQ vs QAT:
    ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    
    Post-Training Quantization (PTQ):
    ✓ No retraining needed
    ✓ Fast and easy
    ✓ Works well for most models
    ✗ May lose 1-3% accuracy on complex models
    
    Quantization-Aware Training (QAT):
    ✓ Better accuracy (simulates quantization during training)
    ✓ Can achieve <1% drop even on complex models
    ✗ Requires retraining (expensive)
    ✗ More complex to implement
    
    
    THE BOTTOM LINE:
    ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    
    Quantization gives you:
     4x smaller models
     2-4x faster inference
     Lower power consumption
     Minimal accuracy loss
    
    It's one of the most effective model optimization techniques!